# **Step 2: Model Training & Checkpoint Saving**
### **Helmet Detection (YOLOv8)**

This notebook covers training and exporting the YOLOv8 model:
1. **Environment Setup**: OpenMP conflict resolution and CUDA GPU detection.
2. **Dataset Verification & Auto-Repair**: Ensures `data.yaml` contains all required YOLO keys (`path`, `train`, `val`, `test`).
3. **Model Loading**: Initialize pretrained YOLOv8 Nano (`yolov8n.pt`).
4. **Model Training**: Fine-tune for 20 epochs with mosaic augmentation and validation caching.
5. **Model Saving**: Export the best weights (`best.pt`) to `saved_models/best.pt`.
6. **Training Telemetry**: Inspect `results.csv` and plot loss curves and mAP progression.


## **1. Import Modules & Hardware Acceleration**

In [1]:
import os
import shutil
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

# Fix OpenMP duplicate library conflict on Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Check GPU acceleration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## **2. Validate & Auto-Configure Dataset (data.yaml)**

In [2]:
BASE_DIR = os.getcwd()
DATA_YAML_PATH = os.path.join(BASE_DIR, 'HelmetDataset', 'data.yaml')
FINAL_DIR = os.path.join(BASE_DIR, 'HelmetDataset')

# Self-healing verification: Ensure all required YOLO keys exist
# (path, train, val, test, nc, names)
yaml_content = {
    'path': FINAL_DIR.replace('\\', '/'),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': ['With Helmet', 'Without Helmet']
}

with open(DATA_YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✅ Verified data.yaml at: {DATA_YAML_PATH}\n")
with open(DATA_YAML_PATH, 'r') as f:
    print(f.read())


✅ Verified data.yaml at: c:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\HelmetDataset\data.yaml

path: c:/My Space/Github_Repo/Traffic Violation Detection-Deep Learning/HelmetDataset
train: train/images
val: valid/images
test: test/images
nc: 2
names:
- With Helmet
- Without Helmet



## **3. Load Pretrained YOLOv8 Base Model**

In [3]:
# Load pretrained YOLOv8n backbone
model = YOLO('yolov8n.pt')
print("✅ YOLOv8n base model loaded successfully.")


✅ YOLOv8n base model loaded successfully.


## **4. Train the YOLOv8 Model**

In [4]:
# Train the model on HelmetDataset
train_results = model.train(
    task="detect",
    data=DATA_YAML_PATH,
    epochs=20,
    batch=16,
    imgsz=640,
    name='train',
    project=os.path.join(BASE_DIR, 'runs', 'detect'),
    cache=True,
    exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)


New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.120  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\HelmetDataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line

## **5. Save Best Model to saved_models/**

In [5]:
# Find and export the best checkpoint to saved_models/best.pt
candidate_paths = [
    os.path.join(BASE_DIR, 'runs', 'detect', 'train', 'weights', 'best.pt'),
    os.path.join(BASE_DIR, 'runs', 'detect', 'runs', 'detect', 'train', 'weights', 'best.pt'),
    os.path.join('runs', 'detect', 'train', 'weights', 'best.pt')
]

BEST_MODEL_PATH = None
for p in candidate_paths:
    if os.path.exists(p):
        BEST_MODEL_PATH = p
        break

if not BEST_MODEL_PATH:
    raise FileNotFoundError("Could not locate best.pt in runs/detect/train/weights/")

SAVED_MODELS_DIR = os.path.join(BASE_DIR, 'saved_models')
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

TARGET_MODEL_PATH = os.path.join(SAVED_MODELS_DIR, 'best.pt')
shutil.copy(BEST_MODEL_PATH, TARGET_MODEL_PATH)

print("=" * 60)
print("✅ MODEL EXPORTED SUCCESSFULLY")
print("=" * 60)
print(f"Exported Location : {TARGET_MODEL_PATH}")
print(f"Model File Size   : {os.path.getsize(TARGET_MODEL_PATH) / (1024 * 1024):.2f} MB")
print("=" * 60)


✅ MODEL EXPORTED SUCCESSFULLY
Exported Location : c:\My Space\Github_Repo\Traffic Violation Detection-Deep Learning\saved_models\best.pt
Model File Size   : 5.96 MB


## **6. Inspect Training Results Telemetry**

In [6]:
# Check training telemetry
csv_candidates = [
    os.path.join(BASE_DIR, 'runs', 'detect', 'train', 'results.csv'),
    os.path.join(BASE_DIR, 'runs', 'detect', 'runs', 'detect', 'train', 'results.csv'),
    os.path.join('runs', 'detect', 'train', 'results.csv')
]

results_csv = None
for cp in csv_candidates:
    if os.path.exists(cp):
        results_csv = cp
        break

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

print("Final 5 Epochs Summary:")
cols = [c for c in ['epoch', 'train/box_loss', 'train/cls_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)'] if c in df.columns]
df[cols].tail()


Final 5 Epochs Summary:


,epoch,train/box_loss,train/cls_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B)
15,16,1.14975,0.81103,0.78060,0.76417,0.80136,0.47955
16,17,1.12974,0.78342,0.75333,0.77424,0.79579,0.47400
17,18,1.13513,0.76030,0.74511,0.78979,0.79946,0.48990
18,19,1.10327,0.75346,0.76057,0.80523,0.82212,0.50186
19,20,1.08895,0.71809,0.79158,0.76092,0.83370,0.50553


## **7. Plot Training Loss Curves & mAP Progression**

In [8]:
# Plot Losses and mAP over Epochs
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss Curves
if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
    axes[0].plot(df["epoch"], df["train/box_loss"], label='Train Box Loss', color='crimson')
    axes[0].plot(df["epoch"], df["val/box_loss"], label='Val Box Loss', color='salmon', linestyle='--')
if 'train/cls_loss' in df.columns and 'val/cls_loss' in df.columns:
    axes[0].plot(df["epoch"], df["train/cls_loss"], label='Train Cls Loss', color='navy')
    axes[0].plot(df["epoch"], df["val/cls_loss"], label='Val Cls Loss', color='royalblue', linestyle='--')
axes[0].set_title("Training & Validation Losses", fontsize=13)
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Loss", fontsize=11)
axes[0].grid(True)
axes[0].legend()

# mAP Curves
if 'metrics/mAP50(B)' in df.columns:
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], marker='o', color='royalblue', label='mAP@50')
if 'metrics/mAP50-95(B)' in df.columns:
    axes[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], marker='s', color='darkorange', label='mAP@50-95')
axes[1].set_title("mAP Progression over Epochs", fontsize=13)
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("mAP", fontsize=11)
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()


<Figure size 1600x500 with 2 Axes>

## **Next Step**
The model has been successfully trained and saved to `saved_models/best.pt`.
👉 Open **`3_test_model.ipynb`** to evaluate performance on the unseen test dataset and test visual predictions!
